## 02: Covariates

Adds the covariates to `cohort_master`: age, sex and education (all participants; with device
type from 01, the covariates of the HC normative model); BMI, Charlson Comorbidity Index (CCI)
and ZIP3 latitude, used for photoperiod (wearable analysis); self-reported race and ethnicity
(descriptive); and psychiatric medication classes (MDD and BD; descriptive, and used to define
the antipsychotic sensitivity analysis).

Time-varying covariates are computed at each task's EtM date (columns suffixed `_{task}`,
used by the models) and at the first EtM date (descriptive). Missing values are kept as `NaN`;
participants are excluded in the modelling notebooks through the `covs_eligible_*` flags
(Task 7).

**Input:** `cohort/cohort_master.parquet` (01) and BigQuery (CDR v9).
**Output:** `covariates/covariates.parquet`; `cohort_master` updated. Run before 03a/03b.
Task 9 runs before Task 7, which assembles everything.

## Task 0: Setup

In [ ]:
import sys
import os
import gc
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.utils import *
from src.cci import compute_cci

OVERWRITE = False

ETM_TASKS = ['gradcpt', 'delaydiscounting', 'flanker', 'emorecog']

# -- Output paths -------------------------------------------------------------
ZIP_OUT      = 'raw_pulls/zip_raw.parquet'
DEMO_OUT     = 'raw_pulls/demographics_raw.parquet'
RACE_ETH_OUT = 'raw_pulls/race_ethnicity_raw.parquet'
EDU_OUT      = 'raw_pulls/education_raw.parquet'
BMI_OUT      = 'raw_pulls/bmi_raw.parquet'
CCI_OUT      = 'raw_pulls/cci_raw.parquet'
MEDS_OUT     = 'raw_pulls/meds_raw.parquet'
COV_OUT      = 'covariates/covariates.parquet'

# -- Medication class constants -----------------------------------------------
MED_CLASSES_LIST = [
    'ssri', 'snri', 'lithium', 'antipsychotic', 'anticonvulsant',
    'maoi', 'tca', 'atypical_antidepressant',
]
MED_ACTIVE_WINDOW = 180  # days; fallback when end_date and days_supply both null

# -- Columns this notebook writes into cohort_master (assembled in Task 7) -----
task_age_cols    = [f'age_at_etm_{t}' for t in ETM_TASKS]
task_bmi_cols    = [f'bmi_{t}'         for t in ETM_TASKS]
task_cci_cols    = [f'cci_{t}'         for t in ETM_TASKS]
task_ab_cols     = [f'covs_eligible_normative_{t}' for t in ETM_TASKS]
task_covsc_cols  = [f'covs_eligible_fitbit_{t}' for t in ETM_TASKS]
med_first_cols   = [f'on_{cls}'            for cls in MED_CLASSES_LIST]
task_med_cols    = [f'on_{cls}_{t}'        for cls in MED_CLASSES_LIST for t in ETM_TASKS]

COV_COLS = (
    ['person_id', 'age_at_etm', 'sex']
    + task_age_cols
    + ['bmi'] + task_bmi_cols
    + ['cci'] + task_cci_cols
    + ['zip3', 'latitude', 'race', 'ethnicity']
    + ['education_raw', 'education_ordinal']
    + ['covs_eligible_normative'] + task_ab_cols
    + ['covs_eligible_fitbit']
    + task_covsc_cols
    + med_first_cols
    + task_med_cols
)

# -- Load cohort spine --------------------------------------------------------
cohort = load_parquet('cohort/cohort_master.parquet')

# cohort_master already carries these if 02 has run before; drop them so the merges below
# do not create _x/_y duplicates and 02 can be re-run without re-running 01.
cohort = cohort.drop(columns=[c for c in COV_COLS if c != 'person_id'], errors='ignore')

cohort['etm_date_first'] = pd.to_datetime(cohort['etm_date_first'])
for task in ETM_TASKS:
    cohort[f'etm_date_{task}'] = pd.to_datetime(cohort[f'etm_date_{task}'])

anchor_map = dict(zip(cohort['person_id'], cohort['etm_date_first']))

# Per-task anchor maps: etm_date_{task} for each task (NaN where task not completed)
task_anchor_maps = {
    task: dict(zip(cohort['person_id'], cohort[f'etm_date_{task}']))
    for task in ETM_TASKS
}

# -- PID lists ----------------------------------------------------------------
all_pids = cohort['person_id'].tolist()
pids_bq  = ','.join(str(p) for p in all_pids)

mdd_pids    = set(cohort.loc[cohort['group'] == 'MDD', 'person_id'])
bd_pids     = set(cohort.loc[cohort['group'] == 'BD',  'person_id'])
clinical_pids    = sorted(mdd_pids | bd_pids)   # MDD + BD — used for clinical-group BQ queries
mdd_pids_bq      = ','.join(str(p) for p in sorted(mdd_pids))
clinical_pids_bq = ','.join(str(p) for p in clinical_pids)

# -- Group masks --------------------------------------------------------------
mdd_mask      = cohort['group'] == 'MDD'
bd_mask       = cohort['group'] == 'BD'
clinical_mask = cohort['group'].isin(['MDD', 'BD'])

# -- Seed incremental attrition log -------------------------------------------
# Rows whose step label starts with one of this notebook's prefixes are dropped and
# regenerated below; rows written by other notebooks are kept.
NB_ATTRITION_PREFIXES = ('NB02 informational: ',)

attrition_rows.clear()
if gcs_exists('cohort/attrition.parquet'):
    _atr = load_parquet('cohort/attrition.parquet')
    _own = (_atr['step'].astype(str)
                        .map(lambda s: s.startswith(NB_ATTRITION_PREFIXES))
                        .astype(bool))
    attrition_rows.extend(_atr[~_own].to_dict('records'))
    print(f'Attrition log: kept {len(attrition_rows)} step(s) from other notebooks; '
          f'dropped {int(_own.sum())} own row(s) for regeneration.')

print(f'Cohort : {len(cohort):,} participants  '
      f'(MDD: {(cohort["group"]=="MDD").sum():,}, '
      f'BD: {(cohort["group"]=="BD").sum():,}, '
      f'HC: {(cohort["group"]=="HC").sum():,})')
print(f'OVERWRITE : {OVERWRITE}')

## Task 1: ZIP Code → Latitude (all participants)

Most recent ZIP code per participant (AoU releases only the first three digits), mapped to
mean latitude with `pgeocode`. 03b uses latitude to compute photoperiod.

In [ ]:
# -- 1a: BQ pull (all participants including HC) -------------------------------
if not gcs_exists(ZIP_OUT) or OVERWRITE:
    zip_raw_df = client.query(f"""
    SELECT
        person_id,
        observation_date,
        value_as_string AS zip_raw
    FROM {tbl('observation')}
    WHERE observation_source_concept_id = 1585250
      AND person_id IN ({pids_bq})
    ORDER BY person_id, observation_date
    """).to_dataframe()
    zip_raw_df['observation_date'] = pd.to_datetime(zip_raw_df['observation_date'])
    save_parquet(zip_raw_df, ZIP_OUT)
    print(f'Queried and cached: {len(zip_raw_df):,} rows')
else:
    zip_raw_df = load_parquet(ZIP_OUT)
    zip_raw_df['observation_date'] = pd.to_datetime(zip_raw_df['observation_date'])
    print(f'Loaded from cache: {len(zip_raw_df):,} rows')

print(f'Persons with any ZIP record : {zip_raw_df["person_id"].nunique():,} / {len(cohort):,}')


In [ ]:
# -- 1b: pgeocode ZIP3 → latitude (all participants) --------------------------
import pgeocode

nomi   = pgeocode.Nominatim('US')
us_df  = nomi._data.copy()
us_df  = us_df[us_df['latitude'].notna()].copy()
us_df['zip3'] = us_df['postal_code'].astype(str).str.strip().str[:3]
us_df  = us_df[us_df['zip3'].str.match(r'^\d{3}$', na=False)]
ZIP3_LAT = us_df.groupby('zip3')['latitude'].mean().to_dict()
print(f'ZIP3 prefixes in lookup table: {len(ZIP3_LAT)}')

# Clean AoU ZIP strings: strip trailing asterisks, keep first 3 digits
zip_df = zip_raw_df.copy()
zip_df['zip3'] = (
    zip_df['zip_raw'].str.replace(r'\*+$', '', regex=True).str.strip().str[:3]
)
valid_mask = (
    zip_df['zip3'].str.match(r'^\d{3}$', na=False)
    & ~zip_df['zip3'].isin(['000'])
)
zip_df = zip_df[valid_mask].copy()
zip_df['latitude'] = zip_df['zip3'].map(ZIP3_LAT)
zip_df = zip_df[zip_df['latitude'].notna()]
print(f'Records with valid ZIP3 + latitude: {len(zip_df):,} | '
      f'Persons: {zip_df["person_id"].nunique():,}')

# Most recent mappable ZIP3 per participant
pid_zip = (
    zip_df.sort_values('observation_date')
    .groupby('person_id')
    .last()
    .reset_index()[['person_id', 'zip3', 'latitude']]
)

# Merge into cohort — all groups; NaN where no mappable ZIP
cohort = cohort.merge(pid_zip, on='person_id', how='left')

for grp in ['MDD', 'BD', 'HC']:
    sub = cohort[cohort['group'] == grp]
    n_total   = len(sub)
    n_lat     = sub['latitude'].notna().sum()
    n_lat_nan = sub['latitude'].isna().sum()
    print(f'\n{grp} with latitude    : {n_lat:,}')
    print(f'{grp} without latitude : {n_lat_nan:,} ({100*n_lat_nan/n_total:.1f}%)')

print('\nLatitude distribution (all groups):')
print(cohort['latitude'].describe().round(2).to_string())

del zip_raw_df, zip_df, pid_zip
gc.collect()


## Task 2: Age and Sex (all participants)

Age in years at each EtM date (from year of birth). Sex: Female or Male; other values are
missing.

In [ ]:
if not gcs_exists(DEMO_OUT) or OVERWRITE:
    demo_df = client.query(f"""
    SELECT
        p.person_id,
        p.year_of_birth,
        gc_sex.concept_name AS sex
    FROM {tbl('person')} p
    LEFT JOIN {tbl('concept')} gc_sex
        ON p.sex_at_birth_concept_id = gc_sex.concept_id
    WHERE p.person_id IN ({pids_bq})
    """).to_dataframe()
    save_parquet(demo_df, DEMO_OUT)
    print(f'Queried and cached: {len(demo_df):,} rows')
else:
    demo_df = load_parquet(DEMO_OUT)
    print(f'Loaded from cache: {len(demo_df):,} rows')

# Age at etm_date_first (retained for EDA / Table 1)
etm_year_map   = dict(zip(cohort['person_id'], cohort['etm_date_first'].dt.year))
demo_df['age_at_etm'] = (
    demo_df['person_id'].map(etm_year_map) - demo_df['year_of_birth']
).astype('Int64')

# Per-task age: age at etm_date_{task} (NaN where task not completed)
for task in ETM_TASKS:
    task_year_map = dict(zip(cohort['person_id'], cohort[f'etm_date_{task}'].dt.year))
    demo_df[f'age_at_etm_{task}'] = (
        demo_df['person_id'].map(task_year_map) - demo_df['year_of_birth']
    ).astype('Int64')

# Sex: binary Female/Male; everything else → NaN
VALID_SEX = {'Female', 'Male'}
raw_sex_counts = demo_df['sex'].value_counts(dropna=False)
demo_df['sex'] = demo_df['sex'].where(demo_df['sex'].isin(VALID_SEX), other=np.nan)

age_merge_cols = ['age_at_etm'] + [f'age_at_etm_{t}' for t in ETM_TASKS]
cohort = cohort.merge(
    demo_df[['person_id'] + age_merge_cols + ['sex']],
    on='person_id', how='left'
)

print('Raw sex values from person table:')
print(raw_sex_counts.to_string())
n_sex_nan = cohort['sex'].isna().sum()
print(f'\nSex NaN after recoding non-M/F: {n_sex_nan:,} ({100*n_sex_nan/len(cohort):.2f}%)')
print(f'\nAge at etm_date_first: mean={cohort["age_at_etm"].mean():.1f}  '
      f'sd={cohort["age_at_etm"].std():.1f}  '
      f'[min={cohort["age_at_etm"].min()}, max={cohort["age_at_etm"].max()}]')
print('\nAge at per-task EtM date (MDD, non-NaN):')
for task in ETM_TASKS:
    col = f'age_at_etm_{task}'
    vals = cohort.loc[cohort['group'] == 'MDD', col].dropna()
    print(f'  {task:<20}: N={len(vals):,}  mean={vals.mean():.1f}  sd={vals.std():.1f}')

del demo_df
gc.collect()

## Task 3: Race and Ethnicity (all participants)

Self-reported at enrolment; descriptive only.

In [ ]:
if not gcs_exists(RACE_ETH_OUT) or OVERWRITE:
    race_eth_df = client.query(f"""
    SELECT
        p.person_id,
        gc_race.concept_name AS race,
        gc_eth.concept_name  AS ethnicity
    FROM {tbl('person')} p
    LEFT JOIN {tbl('concept')} gc_race
        ON p.race_concept_id = gc_race.concept_id
    LEFT JOIN {tbl('concept')} gc_eth
        ON p.ethnicity_concept_id = gc_eth.concept_id
    WHERE p.person_id IN ({pids_bq})
    """).to_dataframe()
    save_parquet(race_eth_df, RACE_ETH_OUT)
    print(f'Queried and cached: {len(race_eth_df):,} rows')
else:
    race_eth_df = load_parquet(RACE_ETH_OUT)
    # Cache may predate the current cohort — restrict to this run's participants.
    race_eth_df = race_eth_df[race_eth_df['person_id'].isin(set(all_pids))]
    print(f'Loaded from cache: {len(race_eth_df):,} rows')

# Merge into cohort — race/ethnicity is queried for all_pids, so HC rows are
# populated too (they feed the HC normative-sample demographics table in 05_summary).
cohort = cohort.merge(race_eth_df, on='person_id', how='left')

for grp in ['MDD', 'BD']:
    print(f'\nRace distribution ({grp}):')
    print(cohort.loc[cohort['group'] == grp, 'race'].value_counts(dropna=False).to_string())
    print(f'\nEthnicity distribution ({grp}):')
    print(cohort.loc[cohort['group'] == grp, 'ethnicity'].value_counts(dropna=False).to_string())

del race_eth_df
gc.collect()

## Task 4: Education (all participants)

Most recent answer to the "The Basics" survey, coded 0 (never attended school) to 7 (advanced
degree); skipped or declined answers are missing.

In [ ]:
EDU_ORDINAL = {
    'HighestGrade_NeverAttended':      0,
    'HighestGrade_OneThroughFour':     1,
    'HighestGrade_FiveThroughEight':   2,
    'HighestGrade_NineThroughEleven':  3,
    'HighestGrade_TwelveOrGED':        4,
    'HighestGrade_CollegeOnetoThree':  5,
    'HighestGrade_CollegeGraduate':    6,
    'HighestGrade_AdvancedDegree':     7,
}

if not gcs_exists(EDU_OUT) or OVERWRITE:
    edu_df = client.query(f"""
    SELECT
        person_id,
        observation_date,
        value_source_value AS education_raw
    FROM {tbl('observation')}
    WHERE observation_source_concept_id = 1585940
      AND person_id IN ({pids_bq})
    """).to_dataframe()
    edu_df['observation_date'] = pd.to_datetime(edu_df['observation_date'])
    save_parquet(edu_df, EDU_OUT)
    print(f'Queried and cached: {len(edu_df):,} rows')
else:
    edu_df = load_parquet(EDU_OUT)
    edu_df['observation_date'] = pd.to_datetime(edu_df['observation_date'])
    print(f'Loaded from cache: {len(edu_df):,} rows')

print(f'Persons with any education record: {edu_df["person_id"].nunique():,}')

# Most recent response per participant
edu_df = (
    edu_df
    .sort_values('observation_date')
    .groupby('person_id', as_index=False)
    .last()
)

# PMI_Skip / PMI_PreferNotToAnswer → NaN
edu_df['education_raw'] = edu_df['education_raw'].replace(
    {'PMI_Skip': np.nan, 'PMI_PreferNotToAnswer': np.nan}
)

# Ordinal encoding 0–7
edu_df['education_ordinal'] = edu_df['education_raw'].map(EDU_ORDINAL).astype('Int64')

cohort = cohort.merge(
    edu_df[['person_id', 'education_raw', 'education_ordinal']],
    on='person_id', how='left',
)

n_edu_nan = cohort['education_ordinal'].isna().sum()
print(f'\nPersons with education_ordinal non-NaN : {cohort["education_ordinal"].notna().sum():,}')
print(f'Persons with education_ordinal NaN     : {n_edu_nan:,} ({100*n_edu_nan/len(cohort):.1f}%)')
print('\nRaw education value counts:')
print(cohort['education_raw'].value_counts(dropna=False).to_string())

del edu_df
gc.collect()

## Task 5: BMI (all participants)

The measurement nearest each EtM date, before or after (measurements on the same day are
averaged).

In [ ]:
# -- 5a: BQ pull (all participants; no date filter — Python applies anchor cutoff below) ---
BMI_CONCEPT_ID = 3038553

if not gcs_exists(BMI_OUT) or OVERWRITE:
    bmi_raw_df = client.query(f"""
    SELECT
        person_id,
        DATE(measurement_date) AS meas_date,
        value_as_number        AS bmi
    FROM {tbl('measurement')}
    WHERE measurement_concept_id = {BMI_CONCEPT_ID}
      AND value_as_number BETWEEN 10 AND 80
      AND person_id IN ({pids_bq})
    """).to_dataframe()
    bmi_raw_df['meas_date'] = pd.to_datetime(bmi_raw_df['meas_date'])
    save_parquet(bmi_raw_df, BMI_OUT)
    print(f'Queried and cached: {len(bmi_raw_df):,} rows')
else:
    bmi_raw_df = load_parquet(BMI_OUT)
    bmi_raw_df['meas_date'] = pd.to_datetime(bmi_raw_df['meas_date'])
    print(f'Loaded from cache: {len(bmi_raw_df):,} rows')

print(f'Persons with any BMI record: {bmi_raw_df["person_id"].nunique():,} / {len(cohort):,}')


In [ ]:
# -- 5b: Nearest-wins BMI selection anchored to etm_date_first (EDA / Table 1) --
# Average same-day measurements: one value per (person_id, meas_date), so the nearest
# date below picks a single BMI regardless of row order (also feeds 5c)
_n_rows = len(bmi_raw_df)
bmi_raw_df = bmi_raw_df.groupby(['person_id', 'meas_date'], as_index=False)['bmi'].mean()
print(f'BMI rows: {_n_rows:,} -> {len(bmi_raw_df):,} person-dates after averaging same-day values')

bmi_raw_df['_anchor'] = bmi_raw_df['person_id'].map(anchor_map)

pre_ref = bmi_raw_df[bmi_raw_df['meas_date'] <= bmi_raw_df['_anchor']].copy()
pre_ref['delta'] = (pre_ref['_anchor'] - pre_ref['meas_date']).dt.days
bmi_pre = (
    pre_ref.sort_values('delta')
    .groupby('person_id')
    .first()[['bmi', 'delta']]
    .reset_index()
    .rename(columns={'bmi': 'bmi_pre', 'delta': 'delta_pre'})
)

post_ref = bmi_raw_df[bmi_raw_df['meas_date'] > bmi_raw_df['_anchor']].copy()
post_ref['delta'] = (post_ref['meas_date'] - post_ref['_anchor']).dt.days
bmi_post = (
    post_ref.sort_values('delta')
    .groupby('person_id')
    .first()[['bmi', 'delta']]
    .reset_index()
    .rename(columns={'bmi': 'bmi_post', 'delta': 'delta_post'})
)

# All participants; no BMI → NaN (excluded later through covs_eligible_fitbit)
all_base = cohort[['person_id']].copy()
bmi_df   = all_base.merge(bmi_pre,  on='person_id', how='left')
bmi_df   = bmi_df.merge(bmi_post, on='person_id', how='left')

both_exist = bmi_df['bmi_pre'].notna() & bmi_df['bmi_post'].notna()
pre_closer = bmi_df['delta_pre'] <= bmi_df['delta_post']
bmi_df['bmi'] = np.where(
    both_exist,
    np.where(pre_closer, bmi_df['bmi_pre'], bmi_df['bmi_post']),
    bmi_df['bmi_pre'].fillna(bmi_df['bmi_post']),
)

cohort = cohort.merge(bmi_df[['person_id', 'bmi']], on='person_id', how='left')

n_only_pre  = int((bmi_df['bmi_pre'].notna() & bmi_df['bmi_post'].isna()).sum())
n_only_post = int((bmi_df['bmi_pre'].isna()  & bmi_df['bmi_post'].notna()).sum())
n_both      = int(both_exist.sum())
print('BMI source breakdown — etm_date_first anchor (all groups):')
print(f'  Pre-reference only  : {n_only_pre:,}')
print(f'  Post-reference only : {n_only_post:,}')
print(f'  Both (nearest used) : {n_both:,}')
for grp in ['MDD', 'BD', 'HC']:
    n_miss = int(cohort.loc[cohort['group'] == grp, 'bmi'].isna().sum())
    n_grp  = int((cohort['group'] == grp).sum())
    print(f'  Missing ({grp}) : {n_miss:,} ({100*n_miss/n_grp:.1f}%)')

# -- 5c: Per-task BMI (all participants; nearest to etm_date_{task}) -----------
print('\nPer-task BMI missingness (all groups):')
for task in ETM_TASKS:
    task_raw = bmi_raw_df.copy()
    task_raw['_anchor'] = task_raw['person_id'].map(task_anchor_maps[task])
    task_raw = task_raw[task_raw['_anchor'].notna()]   # drops non-completers for this task

    pre_t = task_raw[task_raw['meas_date'] <= task_raw['_anchor']].copy()
    pre_t['delta'] = (pre_t['_anchor'] - pre_t['meas_date']).dt.days
    bmi_pre_t = (
        pre_t.sort_values('delta').groupby('person_id').first()[['bmi', 'delta']]
        .reset_index().rename(columns={'bmi': 'bmi_pre', 'delta': 'delta_pre'})
    )

    post_t = task_raw[task_raw['meas_date'] > task_raw['_anchor']].copy()
    post_t['delta'] = (post_t['meas_date'] - post_t['_anchor']).dt.days
    bmi_post_t = (
        post_t.sort_values('delta').groupby('person_id').first()[['bmi', 'delta']]
        .reset_index().rename(columns={'bmi': 'bmi_post', 'delta': 'delta_post'})
    )

    bmi_t  = all_base.merge(bmi_pre_t,  on='person_id', how='left')
    bmi_t  = bmi_t.merge(bmi_post_t, on='person_id', how='left')
    both_t = bmi_t['bmi_pre'].notna() & bmi_t['bmi_post'].notna()
    pre_t2 = bmi_t['delta_pre'] <= bmi_t['delta_post']
    bmi_t[f'bmi_{task}'] = np.where(
        both_t,
        np.where(pre_t2, bmi_t['bmi_pre'], bmi_t['bmi_post']),
        bmi_t['bmi_pre'].fillna(bmi_t['bmi_post']),
    )
    cohort = cohort.merge(bmi_t[['person_id', f'bmi_{task}']], on='person_id', how='left')
    for grp in ['MDD', 'BD', 'HC']:
        n_miss_t = cohort.loc[cohort['group'] == grp, f'bmi_{task}'].isna().sum()
        print(f'  bmi_{task:<20} ({grp}): {n_miss_t:,} missing')

del bmi_raw_df, bmi_pre, bmi_post, bmi_df, pre_ref, post_ref, all_base
gc.collect()


## Task 6: Charlson Comorbidity Index (all participants)

Physical comorbidities recorded on or before each EtM date, scored with the Quan ICD-9-CM and
ICD-10-CM code lists and the original Charlson weights (`src/cci.py`). Psychiatric codes are
excluded. Participants with no matching code have CCI = 0.

In [ ]:
# -- 6a: BQ pull (all participants; no date filter in SQL) ---------------------
if not gcs_exists(CCI_OUT) or OVERWRITE:
    cci_raw_df = client.query(f"""
    SELECT
        co.person_id,
        src.vocabulary_id,
        UPPER(REGEXP_REPLACE(
            co.condition_source_value, r'[^A-Za-z0-9]', '')) AS code,
        MIN(co.condition_start_date)                          AS first_dx_date
    FROM {tbl('condition_occurrence')} co
    LEFT JOIN {tbl('concept')} src
        ON src.concept_id = co.condition_source_concept_id
       AND co.condition_source_concept_id != 0
    WHERE co.person_id IN ({pids_bq})
      AND co.condition_source_value IS NOT NULL
      AND co.condition_source_value NOT LIKE 'F%'
      AND NOT (
            co.condition_source_value LIKE '29%'
         OR co.condition_source_value LIKE '30%'
         OR co.condition_source_value LIKE '31%'
      )
    GROUP BY co.person_id, src.vocabulary_id, code
    """).to_dataframe()
    cci_raw_df['first_dx_date'] = pd.to_datetime(cci_raw_df['first_dx_date'])
    save_parquet(cci_raw_df, CCI_OUT)
    print(f'Queried and cached: {len(cci_raw_df):,} rows')
else:
    cci_raw_df = load_parquet(CCI_OUT)
    cci_raw_df['first_dx_date'] = pd.to_datetime(cci_raw_df['first_dx_date'])
    print(f'Loaded from cache: {len(cci_raw_df):,} rows')

print(f'Persons with any condition record: {cci_raw_df["person_id"].nunique():,}')

# Keep ICD-9-CM / ICD-10-CM source codes only; compute_cci matches each against its own list
_icd = cci_raw_df['vocabulary_id'].isin(['ICD9CM', 'ICD10CM'])
print(f'ICD-9-CM / ICD-10-CM rows kept: {int(_icd.sum()):,} '
      f'(dropped {int((~_icd).sum()):,} rows from other or unmapped source vocabularies)')
cci_raw_df = cci_raw_df[_icd].copy()
print(f'Persons with any ICD condition record: {cci_raw_df["person_id"].nunique():,}')


In [ ]:
# -- 6b: Apply anchor cutoff and compute CCI anchored to etm_date_first (all groups) --
cci_raw_df['_anchor'] = cci_raw_df['person_id'].map(anchor_map)
cci_filtered = cci_raw_df[
    cci_raw_df['first_dx_date'] <= cci_raw_df['_anchor']
].copy()
print(f'Rows after first_dx_date ≤ etm_date_first: {len(cci_filtered):,}')

all_pid_series = cohort['person_id']
cci_scores = compute_cci(cci_filtered, all_pid_series)

cohort = cohort.merge(cci_scores, on='person_id', how='left')
# Fill 0 for all participants (no matching conditions → 0 physical comorbidities)
cohort['cci'] = cohort['cci'].fillna(0).astype('Int64')

for grp in ['MDD', 'BD', 'HC']:
    n_zero = int((cohort.loc[cohort['group'] == grp, 'cci'] == 0).sum())
    n_grp  = int((cohort['group'] == grp).sum())
    print(f'{grp} CCI = 0 (etm_date_first): {n_zero:,} ({100*n_zero/n_grp:.1f}%)')

# -- 6c: Per-task CCI (all groups; conditions up to etm_date_{task}) ----------
# Fill 0 for participants eligible for the task; everyone else stays NaN.
print('\nPer-task CCI (all groups):')
for task in ETM_TASKS:
    task_anchor_col = cci_raw_df['person_id'].map(task_anchor_maps[task])
    cci_task_f = cci_raw_df[
        cci_raw_df['first_dx_date'] <= task_anchor_col
    ].copy()
    cci_task_scores = compute_cci(cci_task_f, all_pid_series)
    cci_task_scores = cci_task_scores.rename(columns={'cci': f'cci_{task}'})
    cohort = cohort.merge(cci_task_scores, on='person_id', how='left')

    task_eligible = (
        (mdd_mask & (cohort[f'mdd_eligible_{task}'].fillna(0) == 1)) |
        (bd_mask  & (cohort[f'bd_eligible_{task}'].fillna(0) == 1)) |
        ((cohort['group'] == 'HC') & cohort[f'etm_date_{task}'].notna())
    )
    cohort.loc[task_eligible, f'cci_{task}'] = (
        cohort.loc[task_eligible, f'cci_{task}'].fillna(0)
    )
    cohort[f'cci_{task}'] = cohort[f'cci_{task}'].astype('Int64')

    for grp in ['MDD', 'BD', 'HC']:
        grp_mask = cohort['group'] == grp
        n_zero_t = int((cohort.loc[grp_mask, f'cci_{task}'] == 0).sum())
        n_nan_t  = int(cohort.loc[grp_mask, f'cci_{task}'].isna().sum())
        print(f'  cci_{task:<20} ({grp}): =0: {n_zero_t:,}  NaN: {n_nan_t:,}')

del cci_raw_df, cci_filtered, cci_scores
gc.collect()


## Task 9: Medications (MDD and BD, descriptive)

Eight classes, each flagged `1.0` if active at the EtM date (`NaN` for HC): SSRI, SNRI,
lithium, antipsychotics, anticonvulsants (lamotrigine, carbamazepine, valproate), MAOI, TCA and
atypical antidepressants (mirtazapine, trazodone, bupropion). Classes are resolved from ATC
groups or RxNorm ingredients through `concept_ancestor`. Antipsychotics are ATC `N05A` minus
lithium (its own class) and minus the phenothiazine antiemetics (e.g. prochlorperazine).

A prescription is active if the EHR end date, or failing that start date + days' supply, or
failing that start date + 180 days, is on or after the EtM date. Prescriptions filled outside
AoU-linked health systems are not captured, so prevalences are lower bounds.

`on_antipsychotic_{task}` defines the exclusion in the antipsychotic sensitivity analysis
(04b); no medication is a model covariate.

In [ ]:
# -- Concept ID mapping -------------------------------------------------------
# ATC groups and RxNorm ingredients, expanded to all products through concept_ancestor.
# Not included: milnacipran (fibromyalgia indication), oxcarbazepine (antiepileptic);
# valproate sodium and divalproex are covered by valproate (745466).
MED_CONCEPTS = [
    ('ssri',                    21604709),  # ATC: SSRIs (N06AB)
    ('lithium',                 21604555),  # ATC: Lithium (N05AN)
    ('antipsychotic',           21604490),  # ATC: Antipsychotics (N05A) MINUS N05AN
                                            #   MINUS the antiemetics below -- see SQL
    ('maoi',                    21604719),  # ATC: Non-selective MAOIs (N06AF)
    ('tca',                     21604687),  # ATC: TCAs (N06AA)
    ('snri',                     715259),   # RxNorm: duloxetine
    ('snri',                     743670),   # RxNorm: venlafaxine
    ('snri',                     717607),   # RxNorm: desvenlafaxine
    ('snri',                   43560354),   # RxNorm: levomilnacipran (Fetzima)
    ('anticonvulsant',           705103),   # RxNorm: lamotrigine
    ('anticonvulsant',           740275),   # RxNorm: carbamazepine
    ('anticonvulsant',           745466),   # RxNorm: valproate (covers divalproex)
    ('atypical_antidepressant',  725131),   # RxNorm: mirtazapine
    ('atypical_antidepressant',  703547),   # RxNorm: trazodone
    ('atypical_antidepressant',  750982),   # RxNorm: bupropion
]

LITHIUM_ANCESTOR = 21604555   # subtracted from 'antipsychotic' only
ANTIEMETIC_EXCLUDE = {        # subtracted from 'antipsychotic' only
    'prochlorperazine', 'droperidol', 'methotrimeprazine', 'promazine',
}

MED_ROSTER_OUT = 'raw_pulls/med_class_roster.parquet'

_concepts_cte = '\n        UNION ALL\n        '.join(
    f"SELECT '{cls}' AS class_label, {cid} AS ancestor_id"
    for cls, cid in MED_CONCEPTS
)
_antiemetic_sql = ', '.join(f"'{a}'" for a in sorted(ANTIEMETIC_EXCLUDE))

# Antipsychotic subtractions (lithium, phenothiazine antiemetics), shared by both queries.
_AP_SUBTRACTIONS = f"""
      AND NOT (
          tc.class_label = 'antipsychotic'
          AND ca.descendant_concept_id IN (SELECT descendant_concept_id FROM lithium_desc)
      )
      AND NOT (
          tc.class_label = 'antipsychotic'
          AND ca.descendant_concept_id IN (SELECT descendant_concept_id FROM antiemetic_desc)
      )"""

_SUBTRACTION_CTES = f"""
lithium_desc AS (
    SELECT descendant_concept_id
    FROM {tbl('concept_ancestor')}
    WHERE ancestor_concept_id = {LITHIUM_ANCESTOR}
),
-- Antiemetic N05A agents, resolved by name rather than hard-coded concept_id so the
-- definition stays readable and survives a vocabulary refresh.
antiemetic_desc AS (
    SELECT ca.descendant_concept_id
    FROM {tbl('concept')} ing
    JOIN {tbl('concept_ancestor')} ca
        ON ca.ancestor_concept_id = ing.concept_id
    WHERE ing.vocabulary_id    = 'RxNorm'
      AND ing.concept_class_id = 'Ingredient'
      AND LOWER(ing.concept_name) IN ({_antiemetic_sql})
),"""


# -- 9a-i: Class roster — what MED_CONCEPTS actually captures ------------------
# The RxNorm ingredients each class resolves to, printed for checking.
if not gcs_exists(MED_ROSTER_OUT) or OVERWRITE:
    med_roster = client.query(f"""
    WITH target_classes AS (
        {_concepts_cte}
    ),{_SUBTRACTION_CTES.rstrip(',')}
    SELECT DISTINCT
        tc.class_label,
        c.concept_id   AS ingredient_concept_id,
        c.concept_name AS ingredient
    FROM target_classes tc
    JOIN {tbl('concept_ancestor')} ca
        ON tc.ancestor_id = ca.ancestor_concept_id
    JOIN {tbl('concept')} c
        ON c.concept_id = ca.descendant_concept_id
    WHERE c.vocabulary_id    = 'RxNorm'
      AND c.concept_class_id = 'Ingredient'{_AP_SUBTRACTIONS}
    ORDER BY tc.class_label, c.concept_name
    """).to_dataframe()
    save_parquet(med_roster, MED_ROSTER_OUT)
    print(f'Queried and cached class roster: {len(med_roster):,} (class, ingredient) pairs')
else:
    med_roster = load_parquet(MED_ROSTER_OUT)
    print(f'Loaded class roster from cache: {len(med_roster):,} (class, ingredient) pairs')

print('\n=== What MED_CONCEPTS resolves to (RxNorm Ingredients, after subtractions) ===')
for cls in MED_CLASSES_LIST:
    ings = sorted(med_roster.loc[med_roster['class_label'] == cls, 'ingredient'].unique())
    print(f'\n-- {cls}  (n={len(ings)}) ' + '-' * max(0, 56 - len(cls)))
    print('   ' + ', '.join(ings) if ings else '   (EMPTY - class definition is broken)')

# Check that both subtractions held.
_ap_roster = set(med_roster.loc[med_roster['class_label'] == 'antipsychotic', 'ingredient']
                 .astype(str).str.lower())
_li_roster = set(med_roster.loc[med_roster['class_label'] == 'lithium', 'ingredient']
                 .astype(str).str.lower())
assert not {i for i in _ap_roster if 'lithium' in i}, 'lithium survived in antipsychotic'
assert not (_ap_roster & ANTIEMETIC_EXCLUDE), \
    f'antiemetics survived in antipsychotic: {sorted(_ap_roster & ANTIEMETIC_EXCLUDE)}'
assert _li_roster and _ap_roster, 'a subtraction emptied its class'
print(f'\nOK - antipsychotic resolves to {len(_ap_roster)} agents, disjoint from lithium '
      f'({len(_li_roster)} agents) and free of antiemetics.')


# -- 9a-ii: BQ pull (MDD and BD; no date filter — Python applies anchor logic) --
# `in_class_ingredient` is null for a product in the ATC group but with none of its agents.
if not gcs_exists(MEDS_OUT) or OVERWRITE:
    meds_raw_df = client.query(f"""
    WITH target_classes AS (
        {_concepts_cte}
    ),{_SUBTRACTION_CTES}
    class_ingredients AS (
        SELECT DISTINCT
            tc.class_label,
            ca.descendant_concept_id AS ingredient_concept_id
        FROM target_classes tc
        JOIN {tbl('concept_ancestor')} ca
            ON tc.ancestor_id = ca.ancestor_concept_id
        JOIN {tbl('concept')} c
            ON c.concept_id = ca.descendant_concept_id
        WHERE c.vocabulary_id    = 'RxNorm'
          AND c.concept_class_id = 'Ingredient'{_AP_SUBTRACTIONS}
    ),
    class_drugs AS (
        SELECT DISTINCT
            tc.class_label,
            ca.descendant_concept_id AS drug_concept_id
        FROM target_classes tc
        JOIN {tbl('concept_ancestor')} ca
            ON tc.ancestor_id = ca.ancestor_concept_id
        WHERE TRUE{_AP_SUBTRACTIONS}
    ),
    drug_in_class_agent AS (
        SELECT DISTINCT
            cd.class_label,
            cd.drug_concept_id,
            ing.concept_name AS in_class_ingredient
        FROM class_drugs cd
        JOIN {tbl('concept_ancestor')} ca3
            ON ca3.descendant_concept_id = cd.drug_concept_id
        JOIN class_ingredients ci
            ON ci.ingredient_concept_id = ca3.ancestor_concept_id
           AND ci.class_label           = cd.class_label
        JOIN {tbl('concept')} ing
            ON ing.concept_id = ca3.ancestor_concept_id
    )
    SELECT
        de.person_id,
        cd.class_label,
        dn.concept_name AS drug_name,
        dia.in_class_ingredient,
        de.drug_exposure_start_date AS start_date,
        de.drug_exposure_end_date   AS end_date,
        de.days_supply
    FROM class_drugs cd
    JOIN {tbl('drug_exposure')} de
        ON de.drug_concept_id = cd.drug_concept_id
    JOIN {tbl('concept')} dn
        ON dn.concept_id = cd.drug_concept_id
    LEFT JOIN drug_in_class_agent dia
        ON dia.class_label     = cd.class_label
       AND dia.drug_concept_id = cd.drug_concept_id
    WHERE de.person_id IN ({clinical_pids_bq})
      AND de.drug_exposure_start_date IS NOT NULL
    """).to_dataframe()
    meds_raw_df['start_date'] = pd.to_datetime(meds_raw_df['start_date'])
    meds_raw_df['end_date']   = pd.to_datetime(meds_raw_df['end_date'])
    for _c in ['class_label', 'drug_name', 'in_class_ingredient']:
        meds_raw_df[_c] = meds_raw_df[_c].astype('category')
    save_parquet(meds_raw_df, MEDS_OUT)
    print(f'\nQueried and cached: {len(meds_raw_df):,} rows')
else:
    meds_raw_df = load_parquet(MEDS_OUT)
    meds_raw_df['start_date'] = pd.to_datetime(meds_raw_df['start_date'])
    meds_raw_df['end_date']   = pd.to_datetime(meds_raw_df['end_date'])
    meds_raw_df = meds_raw_df[meds_raw_df['person_id'].isin(set(clinical_pids))]
    print(f'\nLoaded from cache: {len(meds_raw_df):,} rows')

print(f'Clinical persons with any medication record : {meds_raw_df["person_id"].nunique():,} / {len(clinical_pids):,}')
print(f'\nRows per class:')
print(meds_raw_df['class_label'].value_counts().to_string())


# -- 9a-iii: Observed class composition (no query) -----------------------------
# Person counts < 20 suppressed per AoU policy.
if 'in_class_ingredient' in meds_raw_df.columns:
    print('\n=== Observed composition, distinct clinical persons (any record, ever) ===')
    for cls in MED_CLASSES_LIST:
        sub   = meds_raw_df[meds_raw_df['class_label'] == cls]
        total = sub['person_id'].nunique()
        if total == 0:
            print(f'\n-- {cls}: no exposures in the clinical cohort')
            continue
        print(f'\n-- {cls}  (persons: {total if total >= 20 else "<20 (suppressed)"}) '
              + '-' * max(0, 40 - len(cls)))
        counts = (sub.groupby('in_class_ingredient', observed=True)['person_id']
                     .nunique().sort_values(ascending=False))
        for agent, n in counts.items():
            print(f'   {str(agent):<30} {(f"{n:,}" if n >= 20 else "<20"):>8}'
                  f'  {100 * n / total:5.1f}% of class')
        n_null = sub.loc[sub['in_class_ingredient'].isna(), 'person_id'].nunique()
        if n_null:
            print(f'   {"(no in-class agent)":<30} {(f"{n_null:,}" if n_null >= 20 else "<20"):>8}'
                  f'  {100 * n_null / total:5.1f}% of class')


# -- 9a-iv: Final antipsychotic membership (no query) --------------------------
# Drop antipsychotic records with no in-class agent, and any remaining antiemetic.
_is_ap   = meds_raw_df['class_label'] == 'antipsychotic'
_ap_drop = _is_ap & (
    meds_raw_df['in_class_ingredient'].isna()
    | meds_raw_df['in_class_ingredient'].astype(str).str.lower().isin(ANTIEMETIC_EXCLUDE)
)
meds_for_flags = meds_raw_df[~_ap_drop]
print(f'\nAntipsychotic records dropped (no in-class agent or antiemetic): {int(_ap_drop.sum()):,}')
_left = set(meds_for_flags.loc[meds_for_flags['class_label'] == 'antipsychotic',
                               'in_class_ingredient'].dropna().astype(str).str.lower())
assert not (_left & ANTIEMETIC_EXCLUDE), f'antiemetics in antipsychotic: {_left & ANTIEMETIC_EXCLUDE}'


# -- 9b: Active-at-anchor helper (vectorised) ---------------------------------
def compute_med_flags(meds_df, pid_anchor_map):
    """
    Returns wide DataFrame: person_id + one float64 column per MED_CLASSES_LIST entry.
    1.0 = on (≥1 active record at anchor), 0.0 = not on.
    Participants absent from meds_df are not in the result (→ NaN after left merge).

    Active = start_date ≤ anchor AND eff_end ≥ anchor, where eff_end is:
      tier 1: end_date                (if present)
      tier 2: start_date + days_supply  (if days_supply present, end_date null)
      tier 3: start_date + MED_ACTIVE_WINDOW days  (fallback for both null)
    """
    m = meds_df.copy()
    m['_anchor'] = m['person_id'].map(pid_anchor_map)
    m = m[m['_anchor'].notna() & (m['start_date'] <= m['_anchor'])].copy()

    has_end  = m['end_date'].notna()
    has_days = m['days_supply'].notna() & (m['days_supply'] > 0)

    eff_end = m['start_date'] + pd.Timedelta(days=MED_ACTIVE_WINDOW)           # tier 3 fallback
    eff_end = eff_end.where(~has_days,
        m['start_date'] + pd.to_timedelta(
            m['days_supply'].fillna(0).astype(int), unit='D'))                  # tier 2
    eff_end = eff_end.where(~has_end, m['end_date'])                            # tier 1

    m['is_active'] = eff_end >= m['_anchor']

    active = (
        m[m['is_active']]
        # observed=True: class_label is categorical, and the default would emit every
        # (person, class) pair with a count of 0, which .notna() below reads as "on"
        .groupby(['person_id', 'class_label'], observed=True)
        .size()
        .reset_index(name='_n')
    )
    wide = active.pivot(index='person_id', columns='class_label', values='_n').notna()
    wide.columns.name = None
    for cls in MED_CLASSES_LIST:
        if cls not in wide.columns:
            wide[cls] = False
    wide = wide[MED_CLASSES_LIST].astype(float).reset_index()
    wide = wide.rename(columns={cls: f'on_{cls}' for cls in MED_CLASSES_LIST})
    return wide


# -- 9c: etm_date_first-anchored flags (Table 1 / descriptive) ----------------
med_flags_first = compute_med_flags(meds_for_flags, anchor_map)
cohort = cohort.merge(med_flags_first, on='person_id', how='left')
for cls in MED_CLASSES_LIST:
    col = f'on_{cls}'
    # Clinical participants with no records at all → 0.0 (not on medication)
    cohort.loc[clinical_mask, col] = cohort.loc[clinical_mask, col].fillna(0.0)
    # HC rows stay NaN from the left merge

print('Medication prevalence (etm_date_first anchor):\n')
for grp in ['MDD', 'BD']:
    grp_mask = cohort['group'] == grp
    n_grp    = int(grp_mask.sum())
    print(f'  {grp} (N={n_grp:,}):')
    for cls in MED_CLASSES_LIST:
        col  = f'on_{cls}'
        n_on = int((cohort.loc[grp_mask, col] == 1.0).sum())
        print(f'     on_{cls:<30}: {n_on:,} ({100*n_on/n_grp:.1f}%)')
    print()


# -- 9d: Per-task medication flags (8 classes × 4 tasks = 32 columns) ---------
print('Per-task medication flags (lithium summary shown):')
for task in tqdm(ETM_TASKS, desc='Med flags per task'):
    task_flags = compute_med_flags(meds_for_flags, task_anchor_maps[task])
    task_flags = task_flags.rename(
        columns={f'on_{cls}': f'on_{cls}_{task}' for cls in MED_CLASSES_LIST}
    )
    cohort = cohort.merge(task_flags, on='person_id', how='left')
    for cls in MED_CLASSES_LIST:
        col = f'on_{cls}_{task}'
        cohort.loc[clinical_mask, col] = cohort.loc[clinical_mask, col].fillna(0.0)

    for grp in ['MDD', 'BD']:
        grp_mask = cohort['group'] == grp
        n_grp    = int(grp_mask.sum())
        col      = f'on_lithium_{task}'
        n_on     = int((cohort.loc[grp_mask, col] == 1.0).sum())
        print(f'  on_lithium_{task} ({grp}): {n_on:,} / {n_grp:,} ({100*n_on/n_grp:.1f}%)')

del meds_raw_df, meds_for_flags, med_flags_first
gc.collect()

## Task 7: Assemble and Save

Flags used by the modelling notebooks to define the analytic samples:
- `covs_eligible_normative_{task}`: age, sex, education and device type (`touch_{task}`)
  observed (normative-deviation analysis).
- `covs_eligible_fitbit_{task}`: additionally BMI and latitude observed (wearable analysis).

The versions without `_{task}` use the first EtM date and are descriptive. Photoperiod is
computed per task in 03b.

In [ ]:
mdd_mask      = cohort['group'] == 'MDD'
bd_mask       = cohort['group'] == 'BD'
clinical_mask = cohort['group'].isin(['MDD', 'BD'])

# covs_eligible_normative: age, sex, education all non-NaN (all participants)
cohort['covs_eligible_normative'] = (
    cohort['age_at_etm'].notna()
    & cohort['sex'].notna()
    & cohort['education_ordinal'].notna()
)

# covs_eligible_normative_{task}: also requires touch_{task}, a covariate in the HC normative
# model (otherwise patsy would drop touch-missing rows silently).
for task in ETM_TASKS:
    cohort[f'covs_eligible_normative_{task}'] = (
        cohort['covs_eligible_normative']
        & cohort[f'touch_{task}'].notna()
    )

# covs_eligible_fitbit: etm_date_first-anchored version (all participants)
# True if covs_eligible_normative AND bmi AND latitude all non-NaN
cohort['covs_eligible_fitbit'] = (
    cohort['covs_eligible_normative']
    & cohort['bmi'].notna()
    & cohort['latitude'].notna()
).astype(float)

# covs_eligible_fitbit_{task}: used by the modelling notebooks. Defined for task-eligible
# MDD/BD and HC who completed the task; NaN otherwise.
for task in ETM_TASKS:
    task_eligible = (
        (mdd_mask & (cohort[f'mdd_eligible_{task}'].fillna(0) == 1)) |
        (bd_mask  & (cohort[f'bd_eligible_{task}'].fillna(0) == 1)) |
        ((cohort['group'] == 'HC') & cohort[f'etm_date_{task}'].notna())
    )
    cohort[f'covs_eligible_fitbit_{task}'] = np.where(
        task_eligible,
        (
            cohort[f'covs_eligible_normative_{task}']
            & cohort[f'bmi_{task}'].notna()
            & cohort['latitude'].notna()
        ).astype(float),
        np.nan,
    )

# -- Assemble covariates table ------------------------------------------------
missing_cols = [c for c in COV_COLS if c not in cohort.columns]
if missing_cols:
    print(f'WARNING: missing expected columns: {missing_cols}')

covariates = cohort[COV_COLS].copy()

# Always saved (not OVERWRITE-gated), so covariates.parquet and cohort_master stay in sync.
save_parquet(covariates, COV_OUT)

# -- Update cohort_master: merge covariate columns in, re-save ----------------
cohort_master = load_parquet('cohort/cohort_master.parquet')
cov_merge_cols = [c for c in COV_COLS if c != 'person_id']
cohort_master = cohort_master.drop(
    columns=[c for c in cov_merge_cols if c in cohort_master.columns],
    errors='ignore',
)
cohort_master = cohort_master.merge(covariates, on='person_id', how='left')
save_parquet(cohort_master, 'cohort/cohort_master.parquet')

n_ab_elig = int(cohort['covs_eligible_normative'].sum())
print(f'covariates.parquet    : {len(covariates):,} rows, {len(COV_COLS)-1} covariate columns')
print(f'cohort_master.parquet : {len(cohort_master):,} rows (updated)')
print(f'covs_eligible_normative      : {n_ab_elig:,} / {len(cohort):,}  '
      f'({len(cohort) - n_ab_elig:,} fail)')
for grp in ['MDD', 'BD', 'HC']:
    n_grp    = int((cohort['group'] == grp).sum())
    n_c_elig = int((cohort.loc[cohort['group'] == grp, 'covs_eligible_fitbit'] == 1).sum())
    print(f'covs_eligible_fitbit ({grp}, etm_date_first): {n_c_elig:,} / {n_grp:,}')
print('\ncovs_eligible_normative_{task} per task (touch adds on top of age/sex/education):')
for task in ETM_TASKS:
    n_ab_t = int(cohort[f'covs_eligible_normative_{task}'].sum())
    print(f'  {task:<20}: {n_ab_t:,} / {len(cohort):,}')
print('\ncovs_eligible_fitbit per task (eligible participants):')
for task in ETM_TASKS:
    n_c_t      = int((cohort[f'covs_eligible_fitbit_{task}'] == 1).sum())
    n_elig_mdd = int((cohort[f'mdd_eligible_{task}'].fillna(0) == 1).sum())
    n_elig_bd  = int((cohort[f'bd_eligible_{task}'].fillna(0) == 1).sum())
    n_elig_hc  = int(((cohort['group'] == 'HC') & cohort[f'etm_date_{task}'].notna()).sum())
    print(f'  {task:<20}: {n_c_t:,} '
          f'(MDD-eligible: {n_elig_mdd:,}, BD-eligible: {n_elig_bd:,}, HC: {n_elig_hc:,})')
print(f'\nCovariate dtypes:\n{covariates.dtypes.to_string()}')


## Task 8: QC: Distribution Table and Attrition Log

In [ ]:
mdd_mask      = cohort['group'] == 'MDD'
bd_mask       = cohort['group'] == 'BD'
clinical_mask = cohort['group'].isin(['MDD', 'BD'])

# -- Distribution table -------------------------------------------------------
print('=== Covariate distributions by group ===\n')
for grp in ['MDD', 'BD', 'HC', 'All']:
    sub   = cohort if grp == 'All' else cohort[cohort['group'] == grp]
    n     = len(sub)
    print(f'--- {grp} (N={n:,}) ---')

    age   = sub['age_at_etm'].dropna()
    print(f'  age_at_etm        : mean={age.mean():.1f}  sd={age.std():.1f}  '
          f'[{int(age.min())}–{int(age.max())}]  NaN={sub["age_at_etm"].isna().sum()}')

    n_f   = (sub['sex'] == 'Female').sum()
    n_m   = (sub['sex'] == 'Male').sum()
    n_nan = sub['sex'].isna().sum()
    pct_f = 100 * n_f / (n_f + n_m) if (n_f + n_m) > 0 else float('nan')
    print(f'  sex               : Female={n_f:,} ({pct_f:.1f}% of M+F)  '
          f'Male={n_m:,}  NaN/Other={n_nan:,}')

    edu_ord = sub['education_ordinal'].dropna()
    n_nan   = sub['education_ordinal'].isna().sum()
    print(f'  education_ordinal : mean={edu_ord.mean():.2f}  sd={edu_ord.std():.2f}  '
          f'[{int(edu_ord.min())}–{int(edu_ord.max())}]  NaN={n_nan:,} ({100*n_nan/n:.1f}%)')

    n_ab_pass = int(sub['covs_eligible_normative'].sum())
    print(f'  covs_eligible_normative  : {n_ab_pass:,} / {n:,}  '
          f'(would drop {n - n_ab_pass:,}, {100*(n-n_ab_pass)/n:.1f}%)')

    if grp in ['MDD', 'BD', 'All']:
        sub_clin = sub if grp in ['MDD', 'BD'] else sub[sub['group'].isin(['MDD', 'BD'])]
        n_clin   = len(sub_clin)

        bmi   = sub_clin['bmi'].dropna()
        n_nan = sub_clin['bmi'].isna().sum()
        print(f'  bmi (clinical)    : mean={bmi.mean():.1f}  sd={bmi.std():.1f}  '
              f'[{bmi.min():.1f}–{bmi.max():.1f}]  NaN={n_nan:,} ({100*n_nan/n_clin:.1f}%)')

        cci   = sub_clin['cci'].dropna()
        pct_0 = 100 * (cci == 0).mean()
        print(f'  cci (clinical)    : mean={cci.mean():.2f}  sd={cci.std():.2f}  '
              f'max={cci.max()}  %=0: {pct_0:.1f}%')

        lat   = sub_clin['latitude'].dropna()
        n_nan = sub_clin['latitude'].isna().sum()
        print(f'  latitude (clinical): mean={lat.mean():.2f}  sd={lat.std():.2f}  '
              f'[{lat.min():.2f}–{lat.max():.2f}]  NaN={n_nan:,} ({100*n_nan/n_clin:.1f}%)')

        n_c_pass = int((sub_clin['covs_eligible_fitbit'] == 1).sum())
        print(f'  covs_eligible_fitbit   : {n_c_pass:,} / {n_clin:,}  '
              f'(would drop {n_clin - n_c_pass:,}, {100*(n_clin-n_c_pass)/n_clin:.1f}%)')

    print()

# -- Medication prevalence (MDD and BD; etm_date_first anchor) ----------------
print('=== Medication prevalence (etm_date_first anchor, MDD and BD) ===')
print('(Counts < 20 suppressed per AoU policy)\n')
for grp in ['MDD', 'BD']:
    grp_mask = cohort['group'] == grp
    n_grp    = int(grp_mask.sum())
    print(f'--- {grp} (N={n_grp:,}) ---')
    for cls in MED_CLASSES_LIST:
        col  = f'on_{cls}'
        n_on = int((cohort.loc[grp_mask, col] == 1.0).sum())
        if n_on < 20:
            print(f'  on_{cls:<30}: <20 (suppressed)')
        else:
            print(f'  on_{cls:<30}: {n_on:,} ({100*n_on/n_grp:.1f}%)')
    print()

# -- Attrition log (informational — no rows removed from cohort_master) -------
for grp in ['MDD', 'BD', 'HC']:
    sub    = cohort[cohort['group'] == grp]
    n_base = len(sub)

    # Normative covariates (sex and education; age is never missing)
    for col in ['sex', 'education_ordinal']:
        n_miss = int(sub[col].isna().sum())
        if n_miss > 0:
            log_attrition(
                step=f'NB02 informational: missing {col}',
                group=grp,
                n_before=n_base,
                n_after=n_base - n_miss,
                reason=(
                    f'{n_miss} participants missing {col}. '
                    'Not excluded here; listwise deletion applied in the modelling notebooks.'
                ),
            )

    n_ab_inelig = int((~sub['covs_eligible_normative']).sum())
    log_attrition(
        step='NB02 informational: covs_eligible_normative=False (age/sex/education incomplete)',
        group=grp,
        n_before=n_base,
        n_after=n_base - n_ab_inelig,
        reason=(
            f'{n_ab_inelig} participants missing ≥1 of [age_at_etm, sex, education_ordinal]. '
            'covs_eligible_normative=False excludes them from the normative-deviation analysis.'
        ),
    )

    # Per-task touch covariate, counted among participants eligible for the task only.
    for task in ETM_TASKS:
        if grp == 'MDD':
            task_elig_mask = cohort[f'mdd_eligible_{task}'].fillna(0) == 1
        elif grp == 'BD':
            task_elig_mask = cohort[f'bd_eligible_{task}'].fillna(0) == 1
        else:  # HC
            task_elig_mask = cohort[f'etm_date_{task}'].notna()
        sub_task    = cohort[(cohort['group'] == grp) & task_elig_mask]
        n_task_base = len(sub_task)
        if n_task_base == 0:
            continue

        n_miss_touch = int(sub_task[f'touch_{task}'].isna().sum())
        if n_miss_touch > 0:
            log_attrition(
                step=f'NB02 informational: missing touch_{task}',
                group=grp,
                n_before=n_task_base,
                n_after=n_task_base - n_miss_touch,
                reason=(
                    f'{n_miss_touch} {grp} participants eligible for {task} missing touch_{task}. '
                    'Not excluded here; listwise deletion applied in the modelling notebooks via '
                    f'covs_eligible_normative_{task}.'
                ),
                task=task,
            )

        n_ab_task_inelig = int((~sub_task[f'covs_eligible_normative_{task}']).sum())
        if n_ab_task_inelig > 0:
            log_attrition(
                step=f'NB02 informational: covs_eligible_normative_{task}=False '
                     '(age/sex/education/touch incomplete)',
                group=grp,
                n_before=n_task_base,
                n_after=n_task_base - n_ab_task_inelig,
                reason=(
                    f'{n_ab_task_inelig} {grp} participants eligible for {task} missing ≥1 of '
                    f'[age_at_etm, sex, education_ordinal, touch_{task}]. '
                    f'covs_eligible_normative_{task}=False excludes them from the '
                    f'normative-deviation analysis for {task}.'
                ),
                task=task,
            )

    # Wearable covariates (clinical groups only)
    if grp in ['MDD', 'BD']:
        for col in ['bmi', 'latitude']:
            n_miss = int(sub[col].isna().sum())
            if n_miss > 0:
                log_attrition(
                    step=f'NB02 informational: missing {col}',
                    group=grp,
                    n_before=n_base,
                    n_after=n_base - n_miss,
                    reason=(
                        f'{n_miss} {grp} participants missing {col}. '
                        'Not excluded here; listwise deletion applied in the modelling notebooks.'
                    ),
                )

        n_c_inelig = int((cohort.loc[cohort['group'] == grp, 'covs_eligible_fitbit'] != 1).sum())
        log_attrition(
            step='NB02 informational: covs_eligible_fitbit=False (covs_ab/bmi/latitude incomplete)',
            group=grp,
            n_before=n_base,
            n_after=n_base - n_c_inelig,
            reason=(
                f'{n_c_inelig} {grp} participants missing ≥1 of [covs_eligible_normative, bmi, latitude] (per-anchor covs_eligible_fitbit). '
                'covs_eligible_fitbit=False (descriptive; the wearable analysis gates on the '
                'per-task covs_eligible_fitbit_{task}).'
            ),
        )

# -- Save updated attrition log -----------------------------------------------
atr_df = pd.DataFrame(attrition_rows)
save_parquet(atr_df, 'cohort/attrition.parquet')
print(f'Attrition log updated ({len(atr_df)} total entries) → cohort/attrition.parquet')
print()
print(atr_df[['step', 'group', 'n_before', 'n_dropped', 'n_after']].to_string(index=False))